In [1]:
import numpy as np
import pandas as pd


df = pd.read_csv("../data/raw/may_filtered.csv")
datetimestr = df['date'] + ' ' + df['time']
df['datetime'] = pd.to_datetime(datetimestr, format='%d-%m-%Y %H:%M:%S').astype("datetime64[s]")
df['duration(s)'] = pd.to_timedelta(df.duration).dt.total_seconds()
df.drop(['date', 'time', 'duration'], inplace=True, axis=1)
df['motion'] = np.linalg.norm(df[['accX', 'accY', 'accZ']].values, axis=1)
df = df[["datetime", "duration(s)", "red", "ied", "motion"]]


# 1. 直接在原 df 计算每行 ied 与上一行的差值绝对值
died = df['ied'].diff().abs()

# 2. 处理 0 和 NaN（第一行 diff 会是 NaN），避免 log10 报错
safe_died = np.where((died == 0) | (died.isna()), 1, died)

# 3. 计算原序列的 magnitude，并作为新列加入 df（如果你不想加入 df，也可以作为一个独立的 Series）
df['magnitude'] = np.floor(np.log10(safe_died)).astype(int)

# 4. 找到 magnitude == 6 的那些行的「位置索引」(整数位置，用于分割)
split_positions = np.where(df['magnitude'] == 6)[0]

# 5. 把 df 分割开
# 将首(0)尾(len(df))索引加入列表中，构建出完整的切片区间边界
indices = [0] + split_positions.tolist() + [len(df)]

# 使用 iloc 列表推导式循环切片，确保结果是一个只包含 DataFrame 的列表
df_splits = [df.iloc[indices[i]:indices[i+1]] for i in range(len(indices)-1)]

# 查看分割后的片段数量
print(f"总共找到了 {len(split_positions)} 个 magnitude=6 的切分点。")
print(f"DataFrame 被分割成了 {len(df_splits)} 个部分。")
# 如果你想查看每一段的数据：
# for i, chunk in enumerate(df_splits):
#     print(f"第 {i+1} 段形状: {chunk.shape}")

总共找到了 294 个 magnitude=6 的切分点。
DataFrame 被分割成了 295 个部分。


In [2]:
import pandas as pd

# 确保排序和重置索引
df = df.sort_values('datetime').reset_index(drop=True)

# ==========================================
# 1. 核心修改：找出并剔除 mag=6 相关的两个点
# ==========================================
# 找到 magnitude 为 6 的行
is_mag_6 = df['magnitude'] == 6

# 利用 shift(-1) 把 True 向上平移一行，这样就能同时标记引发突变的前一个点
# fill_value=False 防止最后一行平移后产生 NaN
prev_is_mag_6 = is_mag_6.shift(-1, fill_value=False)

# 取并集（|）：只要是 mag=6 的行，或者它紧挨着的前一行，全部标记为 True
to_drop = is_mag_6 | prev_is_mag_6

# 剔除这些异常点，生成一个完全干净的 DataFrame
df_clean = df[~to_drop].reset_index(drop=True)

print(f"原数据共 {len(df)} 行，剔除了 {to_drop.sum()} 行异常点，剩余 {len(df_clean)} 行。")

# ==========================================
# 2. 对干净的数据进行分割
# ==========================================
# 重新计算干净数据的时间差（刚刚剔除数据的那些地方，自然会产生 >1 秒的时间断层）
time_diff = df_clean['datetime'].diff().dt.total_seconds()

# 判断间断点：因为异常点已经没了，现在只需要单纯根据时间间断（>1秒）来切刀即可
is_gap = time_diff > 1.0

# 使用 cumsum() 累加 True 的个数，生成连续的分组 ID
group_ids = is_gap.cumsum()

# 根据生成的分组 ID 进行 groupby 分割
df_splits = [group for _, group in df_clean.groupby(group_ids)]

# 查看结果
print(f"剔除异常点并按时间间断分割后，总共得到了 {len(df_splits)} 个纯净的 DataFrame 片段。")

# 如果想验证一下每一段的数据量
# for i, chunk in enumerate(df_splits):
#     print(f"第 {i+1} 段包含 {len(chunk)} 行数据")

原数据共 7893800 行，剔除了 554 行异常点，剩余 7893246 行。
剔除异常点并按时间间断分割后，总共得到了 231 个纯净的 DataFrame 片段。


In [3]:
import pandas as pd
import numpy as np
import time

# 用于存储每一段的最终统计结果
summary_results = []

print("开始统计连续片段及 magnitude 分布...")
start_time = time.time()

for i, chunk in enumerate(df_splits):
    # 1. 处理空片段
    if chunk.empty:
        summary_results.append({
            '片段序号': i + 1, '数据行数': 0, '总时长(s)': 0, 
            'mag=4_个数': 0, 'mag=5_个数': 0, 'mag=6_个数': 0
        })
        continue

    # 2. 计算总时长 (复用之前的逻辑)
    dt_values = chunk['datetime'].values 
    unique_dt = np.unique(dt_values) 
    
    if len(unique_dt) < 2:
        total_duration = 0
    else:
        total_duration = (unique_dt[-1] - unique_dt[0]) / np.timedelta64(1, 's')

    # ==========================================
    # 3. 核心统计：提取 magnitude 并快速计数
    # ==========================================
    # 提取为 Numpy 数组以获得极限速度
    mags = chunk['magnitude'].values
    
    # 利用布尔索引求和，瞬间算出各值的个数
    count_4 = np.sum(mags == 4)
    count_5 = np.sum(mags == 5)
    count_6 = np.sum(mags == 6)

    # 4. 记录结果
    summary_results.append({
        '片段序号': i + 1,
        '数据行数': len(chunk),
        '总时长(s)': total_duration,
        'mag=4_个数': count_4,
        'mag=5_个数': count_5,
        'mag=6_个数': count_6
    })

# 转换为 DataFrame
df_mag_summary = pd.DataFrame(summary_results)

print(f"统计完成！耗时: {time.time() - start_time:.2f} 秒\n")

# 打印表格展示前几行
print(df_mag_summary.head())

# 保存为 CSV（防止中文乱码）
df_mag_summary.to_csv("continuous_segments_mag_stats.csv", index=False, encoding="utf-8-sig")

开始统计连续片段及 magnitude 分布...
统计完成！耗时: 0.10 秒

   片段序号    数据行数  总时长(s)  mag=4_个数  mag=5_个数  mag=6_个数
0     1   36780   184.0       930         8         0
1     2  105374   527.0      1543        14         0
2     3  687349  3448.0      4043         4         0
3     4     300     3.0         4         0         0
4     5     280     8.0         3         0         0


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib import rcParams

rcParams['font.family'] = 'SimHei'
rcParams['axes.unicode_minus'] = False

# ==========================================
# 1. 过滤出有效时长 > 3秒 的片段
# ==========================================
valid_chunks = []
for i, chunk in enumerate(df_splits):
    if len(chunk) < 2:
        continue
        
    dt_values = chunk['datetime'].values 
    duration = (np.unique(dt_values)[-1] - np.unique(dt_values)[0]) / np.timedelta64(1, 's')
    
    if duration > 3.0:
        valid_chunks.append({
            'original_idx': i + 1,  
            'duration': duration,
            'data': chunk
        })

print(f"过滤完毕：共有 {len(valid_chunks)} 个片段的时长大于 3 秒。")

if not valid_chunks:
    print("没有找到符合条件的片段！")
else:
    # ==========================================
    # 2. 定义核心绘图函数
    # ==========================================
    def view_segment(segment_idx, start_pct, window_pct):
        chunk_info = valid_chunks[segment_idx]
        orig_idx = chunk_info['original_idx']
        duration = chunk_info['duration']
        df_plot = chunk_info['data']
        
        total_rows = len(df_plot)
        
        # 根据百分比计算需要显示的起止行号
        start_row = int(total_rows * (start_pct / 100.0))
        window_rows = max(1, int(total_rows * (window_pct / 100.0))) 
        
        end_row = start_row + window_rows
        # 防止向右滑动时越界
        if end_row > total_rows:
            end_row = total_rows
            start_row = max(0, end_row - window_rows)
            
        # 切片提取当前窗口的数据
        plot_data = df_plot.iloc[start_row:end_row]
        
        # 开始绘图
        plt.figure(figsize=(15, 5)) 
        
        # 🌟 核心修改区 🌟
        # X 轴不再使用 plot_data['datetime']
        # 而是生成一段从 start_row 到 end_row 的连续整数作为 X 轴
        ied_array = plot_data['ied'].values
        plt.plot(ied_array, color='#1f77b4', linewidth=1)
        
        # 获取当前窗口的起止真实时间，用于在标题中展示
        start_time_str = plot_data['datetime'].iloc[0].strftime('%H:%M:%S')
        end_time_str = plot_data['datetime'].iloc[-1].strftime('%H:%M:%S')
        
        # 动态标题
        plt.title(f"片段序号: {orig_idx} | 时长: {duration:.2f}s | 总行数: {total_rows}\n"
                  f"当前显示: 第 {start_row} 行 至 {end_row} 行 (时间跨度: {start_time_str} - {end_time_str})")
        
        # 将 X 轴的标签也相应修改一下，避免引起歧义
        plt.xlabel("当前窗口的数据点索引 (0, 1, 2...)")
        plt.ylabel("IED 信号 (100Hz 原始高频)")

    # ==========================================
    # 3. 构建 interact 交互面板
    # ==========================================
    segment_slider = widgets.IntSlider(
        min=0, max=len(valid_chunks)-1, step=1, value=0, 
        description='切换片段:', layout=widgets.Layout(width='600px')
    )
    
    start_slider = widgets.FloatSlider(
        min=0, max=100, step=0.1, value=0,  # 步长调小到 0.1，平移更丝滑
        description='左右移动(%):', layout=widgets.Layout(width='600px')
    )
    
    window_slider = widgets.FloatSlider(
        min=0.1, max=100, step=0.1, value=100, # 步长调小到 0.1，支持更精细放大
        description='窗口宽度(%):', layout=widgets.Layout(width='600px')
    )

    widgets.interact(
        view_segment, 
        segment_idx=segment_slider, 
        start_pct=start_slider, 
        window_pct=window_slider
    )

ModuleNotFoundError: No module named 'matplotlib'